<a href="https://colab.research.google.com/github/ShamimtheAnalyst/Crash-Course/blob/main/Missing_Value_Handling_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## __Mastering Missing Data in R: A Systemic Guide__

### এই লেসন থেকে আমরা কী শিখব?

- Detection: ডেটাসেটে কোথায় কোথায় মিসিং ভ্যালু আছে তা খুঁজে বের করা।
- Visualization: গ্রাফের মাধ্যমে মিসিং ডেটার পরিমাণ বোঝা।
- Removal: অপ্রয়োজনীয় মিসিং ডেটা বাদ দেওয়া।
- Imputation: বুদ্ধিমত্তার সাথে খালি জায়গায় নতুন তথ্য (গড় বা মিডিয়ান) বসানো।

### ১. প্রয়োজনীয় লাইব্রেরি ও ডেটা চেক করা
আমরা tidyverse এর পাশাপাশি মিসিং ডেটা দেখার জন্য naniar এবং visdat ব্যবহার করব।

In [ ]:
# Install necessary packages
install.packages(c("naniar", "visdat"))

# প্রয়োজনীয় লাইব্রেরি লোড করা
library(tidyverse)
library(naniar)
library(visdat)

In [ ]:
# Starwars ডেটাসেট দেখা
view(starwars)

In [ ]:
# পুরো ডেটাসেটে মোট কতগুলো NA আছে দেখা
sum(is.na(starwars))

In [ ]:
# প্রতিটি কলামে আলাদাভাবে কয়টি করে NA আছে দেখা
colSums(is.na(starwars))

__ব্যাখ্যা:__ is.na() প্রতিটি ভ্যালু চেক করে। কিন্তু বড় ডেটাসেটে এটি পড়া কঠিন, তাই আমরা colSums() ব্যবহার করি যা এক নজরে বলে দেয় কোন কলামে কয়টি তথ্য নেই।

### ২. মিসিং ডেটা ভিজ্যুয়ালাইজ করা (Visual Overview)
কোড দেখে বোঝা কঠিন হতে পারে, তাই গ্রাফ ব্যবহার করা বুদ্ধিমত্তার কাজ।

In [ ]:
# পুরো ডেটাসেটের কোথায় কোথায় তথ্য নেই তার একটি ম্যাপ
vis_miss(starwars)

In [ ]:
# কোন ভ্যারিয়েবলে সবচেয়ে বেশি মিসিং ডেটা আছে তার চার্ট
gg_miss_var(starwars)

__ব্যাখ্যা:__ এই গ্রাফগুলো থেকে আমরা বুঝতে পারি birth_year বা mass কলামে অনেক তথ্য নেই, তাই এগুলো নিয়ে কাজ করার সময় আমাদের সাবধান হতে হবে।

### ৩. মিসিং ডেটা দূর করা (Removal)
যদি কোনো রো-তে গুরুত্বপূর্ণ তথ্য না থাকে, তবে আমরা সেই রো-টি বাদ দিতে পারি।

In [ ]:
# যে কোনো কলামে NA থাকলে সেই সারি মুছে ফেলা
starwars_clean <- starwars %>%
  drop_na()

# চেক করা (উচ্চতার আর কোনো NA নেই)
sum(is.na(starwars_clean$height))

In [ ]:
# শুধুমাত্র উচ্চতা (height) কলামে NA থাকলে সেই সারি মুছে ফেলা
starwars_height_clean <- starwars %>%
  drop_na(height)

# চেক করা (উচ্চতার আর কোনো NA নেই)
sum(is.na(starwars_height_clean$height))

__ব্যাখ্যা:__ drop_na() খুবই শক্তিশালী। তবে সাবধান! এটি ব্যবহার করলে অনেক সময় দামী তথ্য হারিয়ে যেতে পারে। তাই নির্দিষ্ট কলাম উল্লেখ করে বাদ দেওয়া (Option 2) নিরাপদ।

### ৪. মিসিং ডেটা পূরণ করা (Imputation)
তথ্য ডিলিট না করে খালি জায়গায় নতুন কোনো ভ্যালু বসানোকে বলে Imputation।

#### A. নির্দিষ্ট ভ্যালু দিয়ে পূরণ (Replacement)

In [ ]:
starwars_clean <- starwars %>%
  replace_na(list(height = 0, mass = 0))

# চেক করা (উচ্চতার আর কোনো NA নেই)
sum(is.na(starwars_clean$height))

__ব্যাখ্যা:__ এখানে আমরা বলছি যেখানে উচ্চতা নেই সেখানে 0 বসিয়ে দাও।

#### B. গড় দিয়ে পূরণ (Advanced Mutate)
এটি সবচেয়ে প্রফেশনাল পদ্ধতি। আমরা খালি জায়গায় সেই কলামের গড় (Mean) মান বসিয়ে দেব।

In [ ]:
starwars_clean <- starwars %>%
  mutate(height = ifelse(is.na(height),
                         mean(height, na.rm = TRUE),
                         height))

# চেক করা (উচ্চতার আর কোনো NA নেই)
sum(is.na(starwars_mean_fill$height))

__ব্যাখ্যা:__ ifelse ফাংশনটি চেক করছে—যদি উচ্চতা NA হয়, তবে সেটিকে mean (গড়)/ দিয়ে বদলাও, আর যদি NA না হয় তবে যা আছে তাই রাখো। na.rm = TRUE না দিলে কিন্তু গড় বের হবে না।

#### অপশন ৩: মধ্যক (Median) দিয়ে পূরণ করা (Imputation)
এটি প্রফেশনাল পদ্ধতি যেখানে খালি ঘরে সেই কলামের মধ্যক মান বসানো হয়।

In [ ]:
# উচ্চতার খালি ঘরে মধ্যক (Median) মান বসানো
starwars_median_fill <- starwars %>%
  mutate(height = ifelse(is.na(height),
                         mean(height, na.rm = TRUE),
                         height))

# চেক করা (উচ্চতার আর কোনো NA নেই)
sum(is.na(starwars_median_fill$height))

### ৫. ফাইনাল রিপোর্ট ও ভিজ্যুয়ালাইজেশন (Before vs After)
সবশেষে আমরা দেখব আমাদের ক্লিনিং কাজ করল কিনা।

In [ ]:
# ক্লিনিং করার পর টপ ১০ লম্বা ক্যারেক্টারদের চার্ট
starwars %>%
  drop_na(height) %>%   # চার্ট করার আগে NA ফেলে দেওয়া জরুরি
  filter(height > 200) %>%
  ggplot(aes(x = reorder(name, height), y = height)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  labs(title = "Top Tallest Characters", x = "Character Name", y = "Height (cm)") +
  theme_minimal()

### সিস্টেমিক চেকলিস্ট (Cheat Sheet):

- খুঁজতে: colSums(is.na(data))
- দেখতে: vis_miss(data)
- ফেলে দিতে: drop_na(column)
- পূরণ করতে: mutate(col = ifelse(is.na(col), mean, col))